# **DATA COLLECTION STAGE**
> The for this research can be collected from the existing literature on **OpenAlex** website. This website is an excellent landing place for scientific research. It's a huge, free scholarly database. 

In [4]:
# importing libraries

import pandas as pd
import requests

In [5]:
url = "https://api.openalex.org/works?search=Explainable+AI+Infrastructure&filter=publication_year:2018-2026,type:article"
response = requests.get(url)
data = response.json()

In [11]:
def reconstruct_abstract(inverted_index):
    """
    Convert OpenAlex abstract_inverted_index to readable text.
    """
    if not inverted_index:
        return None

    length = max(max(pos) for pos in inverted_index.values()) + 1
    abstract = [""] * length

    for word, positions in inverted_index.items():
        for pos in positions:
            abstract[pos] = word

    return " ".join(abstract)


def get_authors(authorships):
    """
    Return all author names separated by semicolons.
    """
    if not authorships:
        return None

    return "; ".join(
        author.get("author", {}).get("display_name", "")
        for author in authorships
        if author.get("author")
    )


def get_first_author(authorships):
    """
    Return the first author.
    """
    if not authorships:
        return None

    return authorships[0].get("author", {}).get("display_name")


def get_num_authors(authorships):
    """
    Return number of authors.
    """
    return len(authorships) if authorships else 0


def get_institutions(authorships):
    """
    Return all institutions.
    """
    if not authorships:
        return None

    institutions = set()

    for author in authorships:
        for inst in author.get("institutions", []):
            name = inst.get("display_name")
            if name:
                institutions.add(name)

    return "; ".join(sorted(institutions))


def get_countries(authorships):
    """
    Return all countries represented in the paper.
    """
    if not authorships:
        return None

    countries = set()

    for author in authorships:
        for inst in author.get("institutions", []):
            country = inst.get("country_code")
            if country:
                countries.add(country)

    return "; ".join(sorted(countries))


def get_keywords(keywords):
    """
    Return keywords.
    """
    if not keywords:
        return None

    return "; ".join(
        kw.get("display_name")
        for kw in keywords
        if kw.get("display_name")
    )


def get_num_keywords(keywords):
    return len(keywords) if keywords else 0


def get_concepts(concepts):
    """
    Return concepts.
    """
    if not concepts:
        return None

    return "; ".join(
        concept.get("display_name")
        for concept in concepts
        if concept.get("display_name")
    )


def get_top_concepts(concepts, score_threshold=0.5):
    """
    Return concepts with score above threshold.
    """
    if not concepts:
        return None

    selected = [
        c["display_name"]
        for c in concepts
        if c.get("score", 0) >= score_threshold
    ]

    return "; ".join(selected)


def get_referenced_works(referenced_works):
    """
    Return referenced OpenAlex IDs.
    """
    if not referenced_works:
        return None

    return "; ".join(referenced_works)


def get_journal(paper):
    """
    Return journal name.
    """
    return (
        paper.get("primary_location", {})
        .get("source", {})
        .get("display_name")
    )


def get_publisher(paper):
    """
    Return publisher.
    """
    return (
        paper.get("primary_location", {})
        .get("source", {})
        .get("host_organization_name")
    )


def is_open_access(paper):
    """
    Return Open Access status.
    """
    return paper.get("open_access", {}).get("is_oa")


def get_pdf_url(paper):
    """
    Return PDF URL if available.
    """
    return (
        paper.get("open_access", {})
        .get("oa_url")
    )


def get_landing_page(paper):
    """
    Return landing page URL.
    """
    return (
        paper.get("primary_location", {})
        .get("landing_page_url")
    )


def get_source_type(paper):
    """
    Journal, conference, repository, etc.
    """
    return (
        paper.get("primary_location", {})
        .get("source", {})
        .get("type")
    )

In [12]:
papers = []

for paper in data["results"]:

    papers.append({

        "id": paper.get("id"),
        "doi": paper.get("doi"),

        "title": paper.get("title"),
        "abstract": reconstruct_abstract(
            paper.get("abstract_inverted_index")
        ),

        "publication_year": paper.get("publication_year"),
        "publication_date": paper.get("publication_date"),

        "citation_count": paper.get("cited_by_count"),

        "journal": get_journal(paper),
        "publisher": get_publisher(paper),
        "source_type": get_source_type(paper),

        "authors": get_authors(paper.get("authorships")),
        "first_author": get_first_author(paper.get("authorships")),
        "num_authors": get_num_authors(paper.get("authorships")),

        "institutions": get_institutions(paper.get("authorships")),
        "countries": get_countries(paper.get("authorships")),

        "keywords": get_keywords(paper.get("keywords")),
        "num_keywords": get_num_keywords(paper.get("keywords")),

        "concepts": get_concepts(paper.get("concepts")),
        "top_concepts": get_top_concepts(paper.get("concepts")),

        "referenced_works": get_referenced_works(
            paper.get("referenced_works")
        ),

        "open_access": is_open_access(paper),
        "pdf_url": get_pdf_url(paper),
        "landing_page": get_landing_page(paper),
    })

In [13]:
df = pd.DataFrame(papers)

In [14]:
print("Total number of papers found:", len(df))

Total number of papers found: 25


In [15]:
df.head()

,id,doi,title,abstract,publication_year,publication_date,citation_count,journal,publisher,source_type,...,institutions,countries,keywords,num_keywords,concepts,top_concepts,referenced_works,open_access,pdf_url,landing_page
0,https://openalex.org/W2896457183,https://doi.org/10.4230/lipics.cosit.2022.18,AI-Assisted Pipeline for Dynamic Generation of...,Although geospatial question answering systems...,2018,2018-10-11,45745,DROPS (Schloss Dagstuhl – Leibniz Center for I...,Schloss Dagstuhl – Leibniz Center for Informatics,repository,...,Eindhoven University of Technology; Google (Un...,GR; NL; US,Transformer; Computer science; Training (meteo...,8,Transformer; Computer science; Training (meteo...,Transformer,https://openalex.org/W131533222; https://opena...,True,https://drops.dagstuhl.de/entities/document/10...,https://drops.dagstuhl.de/entities/document/10...
1,https://openalex.org/W4292779060,https://doi.org/10.4230/lipics.giscience.2023.43,Aion Framework: Dimensional Emergence of AI Co...,"The Aion Framework presents a bold, unified di...",2023,2023-01-01,14264,Leibniz-Zentrum für Informatik (Schloss Dagstuhl),Schloss Dagstuhl – Leibniz Center for Informatics,repository,...,American Anthropological Association,US,Computer science; Task (project management); L...,9,Computer science; Task (project management); L...,Computer science; Task (project management); L...,None,True,https://drops.dagstuhl.de/storage/00lipics/lip...,https://drops.dagstuhl.de/entities/document/10...
2,https://openalex.org/W2962772482,https://doi.org/10.1145/3236009,A survey of methods for explaining black box m...,"In recent years, many accurate decision suppor...",2019,2019-01-01,4812,ISTI Open Portal,None,repository,...,Istituto di Scienza e Tecnologie dell'Informaz...,IT,Interpretability; Black box; Computer science;...,8,Interpretability; Black box; Computer science;...,Interpretability; Black box; Computer science;...,https://openalex.org/W1531743498; https://open...,True,https://openportal.isti.cnr.it/doc?id=people__...,https://openportal.isti.cnr.it/doc?id=people__...
3,https://openalex.org/W2902634493,https://doi.org/10.1007/s11023-018-9482-5,AI4People—An Ethical Framework for a Good AI S...,This article reports the findings of AI4People...,2018,2018-11-25,3707,Minds and Machines,Springer Science+Business Media,journal,...,Agence Nationale de la Recherche; Bocconi Univ...,BE; BY; CH; DE; FR; GB; IT; NL; SE; US,Philosophy of science; Foundation (evidence); ...,13,Philosophy of science; Foundation (evidence); ...,Philosophy of science; Foundation (evidence); ...,https://openalex.org/W598851794; https://opena...,True,https://link.springer.com/content/pdf/10.1007/...,https://doi.org/10.1007/s11023-018-9482-5
4,https://openalex.org/W2969625533,https://doi.org/10.1016/j.ijinfomgt.2019.08.002,Artificial Intelligence (AI): Multidisciplinar...,"As far back as the industrial revolution, sign...",2019,2019-08-27,4139,International Journal of Information Management,Elsevier BV,journal,...,Aston University; Capgemini (United Kingdom); ...,DK; GB; IN; NL; US,Pace; Transformative learning; Government (lin...,12,Pace; Transformative learning; Government (lin...,Pace; Transformative learning; Government (lin...,https://openalex.org/W23329852; https://openal...,True,https://www.sciencedirect.com/science/article/...,https://doi.org/10.1016/j.ijinfomgt.2019.08.002


In [16]:
df.to_csv('xai_infstr.csv', index=False)